# 09 · Final pipeline: tiled inference, flip TTA, model fusion

Takes trained models from 07 (YOLO) and 08 (RT-DETR) and tests, one step at a time on the val frames with the shared evaluator,
what inference-time additions help:

| Step | What | Why |
|---|---|---|
| full | One model on the whole frame, as in 07 / 08 | Reference |
| + flip | Also predict the horizontally flipped frame; merge with class-wise NMS | Cheap test-time augmentation |
| + tiles | Also predict 640 px tiles (20% overlap) at the model's 640 input, shift boxes back, merge | Small objects are the bottleneck (more than half are missed); a tile shows them at native resolution instead of downscaled |
| + tiles (small only) | Only tile boxes smaller than 64 px are added | Tiles cut large objects into pieces; the full frame handles those |
| WBF | Weighted box fusion of several models' boxes | YOLO and RT-DETR fail differently (YOLO better on small objects, RT-DETR on large ones and recall) |

Every model's passes are computed once and cached (`runs/pipeline/<model>.pkl`), so all combinations are scored without re-running models.
The last section writes the test submission with the chosen configuration.

## Setup

In [1]:
import json
import os
import pickle
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

SRC_DIR = Path(os.environ.get("SRC_DIR", Path.cwd()))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "../multiple-object-video-detection-hw-2-ucu-ai-2026"))
PROC_ROOT = Path(os.environ.get("PROC_ROOT", "../data_processed"))
RUNS_ROOT = Path(os.environ.get("RUNS_ROOT", "../runs"))
sys.path.insert(0, str(SRC_DIR))
from ultralytics import RTDETR, YOLO

from common.evaluate import evaluate_detections
from common.pipeline import TileCfg, cache_passes, combine, fuse, predict_config, to_coco, to_submission_rows
from common.submission import write_submission
from common.yolo import val_items

FIXTURE = os.environ.get("FIXTURE") == "1"
GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
OUT = RUNS_ROOT / ("pipeline_fixture" if FIXTURE else "pipeline")
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

device: NVIDIA GeForce RTX 4060 Laptop GPU


## Models and tiling

Candidates: the best runs per family, with the full-frame size each was trained at. Runs that don't exist (yet) are skipped,
so the final long runs (`final_*`) can be added as they finish.

In [2]:
CANDIDATES = {   # name: (class, run folder, full-frame imgsz)
    "yolo_low_scale_960": (YOLO, RUNS_ROOT / "yolo/low_scale_960", 960),
    "yolo_baseline_1280": (YOLO, RUNS_ROOT / "yolo/baseline", 1280),
    "yolo_final_960":     (YOLO, RUNS_ROOT / "yolo/final_960", 960),
    "yolo_final_1280":    (YOLO, RUNS_ROOT / "yolo/final_1280", 1280),
    "rtdetr_960":         (RTDETR, RUNS_ROOT / "rtdetr/imgsz_960", 960),
    "rtdetr_final_960":   (RTDETR, RUNS_ROOT / "rtdetr/final_960", 960),
    "rtdetr_final_960_s025": (RTDETR, RUNS_ROOT / "rtdetr/final_960_s025", 960),
}
MODELS = {n: v for n, v in CANDIDATES.items() if (v[1] / "best.pt").exists()}
print("available:", list(MODELS))

TILES = TileCfg(size=640, overlap=0.2, imgsz=640, conf=0.01, min_side=1000)
TILE_MAX_SIDE = 64            # "small only": tile boxes whose longer side is < 64 px (64 beat 96 in the CPU ablation)
WBF_IOUS = (0.55, 0.6, 0.65)  # box-matching IoU for weighted box fusion
items = val_items(GT_FILE, DATA_ROOT, max_images=6 if FIXTURE else None)
image_ids = [i for i, _ in items]

available: ['yolo_low_scale_960', 'yolo_baseline_1280', 'yolo_final_960', 'yolo_final_1280', 'rtdetr_960', 'rtdetr_final_960', 'rtdetr_final_960_s025']


## Cache every model's passes on val (full frame, flip, tiles)

On the RTX 4060 this takes a few minutes per YOLO model and longer for RT-DETR. It resumes if interrupted.

In [3]:
caches = {}
for name, (cls, run, imgsz) in MODELS.items():
    print(name)
    caches[name] = cache_passes(cls(str(run / "best.pt")), items, imgsz, TILES, OUT / f"val_{name}.pkl", device=DEVICE)

yolo_low_scale_960


  val_yolo_low_scale_960.pkl: 25/359 images, 0.1 s/image


  val_yolo_low_scale_960.pkl: 50/359 images, 0.1 s/image


  val_yolo_low_scale_960.pkl: 75/359 images, 0.1 s/image


  val_yolo_low_scale_960.pkl: 100/359 images, 0.1 s/image


  val_yolo_low_scale_960.pkl: 125/359 images, 0.1 s/image


... (102 outputs left out)

  val_rtdetr_final_960_s025.pkl: 300/359 images, 0.8 s/image


  val_rtdetr_final_960_s025.pkl: 325/359 images, 0.8 s/image


  val_rtdetr_final_960_s025.pkl: 350/359 images, 0.8 s/image


  val_rtdetr_final_960_s025.pkl: 359/359 images, 0.8 s/image


## Ablation: one model, one step at a time

In [4]:
def score(fn):
    dets = []
    for i in image_ids:
        dets += to_coco(i, fn(i))
    return evaluate_detections(GT_FILE, dets, image_ids=image_ids)

VARIANTS = {
    "full":                  dict(passes=("full",)),
    "+flip":                 dict(passes=("full", "flip")),
    "+tiles":                dict(passes=("full", "tiles")),
    "+tiles small":          dict(passes=("full", "tiles"), tile_max_side=TILE_MAX_SIDE),
    "+tiles small+flip":     dict(passes=("full", "flip", "tiles"), tile_max_side=TILE_MAX_SIDE),
    "+tiles+flip":           dict(passes=("full", "flip", "tiles")),
}
rows = {}
for name in MODELS:
    for label, v in VARIANTS.items():
        rows[(name, label)] = score(lambda i, c=caches[name], v=v: combine(c[i], v["passes"], tile_max_side=v.get("tile_max_side")))
cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
single = pd.DataFrame(rows).T[cols]
single.style.format("{:.4f}").highlight_max(axis=0)

**Observations:** _fill in_

## Fusion of several models (WBF)

Each member uses its best single-model variant from the table above; the members' boxes are then fused with weighted box fusion.

In [5]:
best_variant = {n: single.loc[n]["mAP"].idxmax() for n in MODELS}
print("best variant per model:", best_variant)

def member(n, label=None):
    v = VARIANTS[label or best_variant[n]]
    return lambda i: combine(caches[n][i], v["passes"], tile_max_side=v.get("tile_max_side"))

yolos = [n for n in MODELS if n.startswith("yolo")]
detrs = [n for n in MODELS if n.startswith("rtdetr")]
best_yolo = max(yolos, key=lambda n: single.loc[(n, best_variant[n]), "mAP"]) if yolos else None
best_detr = max(detrs, key=lambda n: single.loc[(n, best_variant[n]), "mAP"]) if detrs else None
GROUPS = {"best YOLO + best RT-DETR": [n for n in (best_yolo, best_detr) if n],
          "all YOLO": yolos, "all models": list(MODELS)}
fused = {}
for label, names in GROUPS.items():
    if len(names) < 2:
        continue
    for tag, use_best in [("full frames", False), ("best variants", True)]:
        fns = [member(n) if use_best else member(n, "full") for n in names]
        for wiou in WBF_IOUS:
            fused[f"{label}, {tag}, iou {wiou}"] = score(
                lambda i, fns=fns, n0=names[0], wiou=wiou: fuse([f(i) for f in fns], caches[n0][i]["shape"], iou=wiou))
pd.DataFrame(fused).T[cols].style.format("{:.4f}").highlight_max(axis=0)

best variant per model: {'yolo_low_scale_960': 'full', 'yolo_baseline_1280': '+flip', 'yolo_final_960': '+flip', 'yolo_final_1280': '+flip', 'rtdetr_960': '+flip', 'rtdetr_final_960': '+flip', 'rtdetr_final_960_s025': '+flip'}


,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR
"best YOLO + best RT-DETR, full frames, iou 0.55",0.2546,0.4825,0.2385,0.0986,0.2098,0.4494,0.4500
"best YOLO + best RT-DETR, full frames, iou 0.6",0.2583,0.4862,0.2450,0.1011,0.2142,0.4525,0.4574
"best YOLO + best RT-DETR, full frames, iou 0.65",0.2609,0.4873,0.2469,0.1027,0.2175,0.4521,0.4628
"best YOLO + best RT-DETR, best variants, iou 0.55",0.2616,0.4888,0.2459,0.1061,0.2133,0.4573,0.4651
"best YOLO + best RT-DETR, best variants, iou 0.6",0.2664,0.4967,0.2513,0.1107,0.2188,0.4599,0.4694
"best YOLO + best RT-DETR, best variants, iou 0.65",0.2691,0.5004,0.2547,0.1131,0.2221,0.4609,0.4709
"all YOLO, full frames, iou 0.55",0.2507,0.4718,0.2359,0.1407,0.2201,0.4162,0.4048
"all YOLO, full frames, iou 0.6",0.2534,0.4760,0.2392,0.1425,0.2233,0.4154,0.4100
"all YOLO, full frames, iou 0.65",0.2570,0.4789,0.2436,0.1468,0.2269,0.4175,0.4154
"all YOLO, best variants, iou 0.55",0.2562,0.4803,0.2393,0.1390,0.2255,0.4169,0.4177


**Observations:** _fill in_

## Choose the final configuration

Set `FINAL` from the two tables (members and each member's variant). The default picks the best single-model or fused row automatically.

In [6]:
best_single = max(((n, l) for n, l in rows), key=lambda k: rows[k]["mAP"])
best_fused = max(fused, key=lambda k: fused[k]["mAP"]) if fused else None
FINAL_WBF_IOU = WBF_IOUS[0]
if best_fused and fused[best_fused]["mAP"] > rows[best_single]["mAP"]:
    label = best_fused
    group, tag, iou_tag = [t.strip() for t in label.split(",")]
    names = GROUPS[group]
    FINAL = {n: VARIANTS[best_variant[n] if tag == "best variants" else "full"] for n in names}
    FINAL_WBF_IOU = float(iou_tag.split()[1])
    print(f"final = fusion '{label}': val mAP {fused[label]['mAP']:.4f}")
else:
    FINAL = {best_single[0]: VARIANTS[best_single[1]]}
    print(f"final = single {best_single}: val mAP {rows[best_single]['mAP']:.4f}")
print(FINAL)
_ = (OUT / "final_config.json").write_text(json.dumps(
    {"wbf_iou": FINAL_WBF_IOU, "members": {n: {**v, "passes": list(v["passes"])} for n, v in FINAL.items()}}, indent=1))

final = fusion 'all models, best variants, iou 0.65': val mAP 0.2970
{'yolo_low_scale_960': {'passes': ('full',)}, 'yolo_baseline_1280': {'passes': ('full', 'flip')}, 'yolo_final_960': {'passes': ('full', 'flip')}, 'yolo_final_1280': {'passes': ('full', 'flip')}, 'rtdetr_960': {'passes': ('full', 'flip')}, 'rtdetr_final_960': {'passes': ('full', 'flip')}, 'rtdetr_final_960_s025': {'passes': ('full', 'flip')}}


## Test submission

Runs the chosen configuration on every test frame (≈ 3,800; about 20–60 min depending on the passes) and writes `runs/pipeline/submission.csv`.

In [7]:
members = {n: (MODELS[n][0](str(MODELS[n][1] / "best.pt")), MODELS[n][2]) for n in FINAL}
test = pd.read_csv(TEST_CSV)
if FIXTURE:
    test = test.head(2)
preds = {}
for k, (key, fname) in enumerate(zip(test["image_id"], test["file_name"]), 1):
    img = cv2.imread(str(DATA_ROOT / fname))
    preds[key] = to_submission_rows(predict_config(members, img, TILES, FINAL, wbf_iou=FINAL_WBF_IOU, device=DEVICE))
    if k % 200 == 0:
        print(f"{k}/{len(test)}")
sub = write_submission(preds, TEST_CSV, OUT / "submission.csv")
n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) // 6
classes = {int(t) for s in sub.PredictionString.dropna() for t in s.split()[0::6]}
assert classes <= set(range(8)) and n_boxes.max() <= 500
print(f"wrote {OUT / 'submission.csv'}: {len(sub)} rows, {int((n_boxes > 0).sum())} with detections, max {int(n_boxes.max())} boxes")

200/3811


400/3811


600/3811


800/3811


1000/3811


1200/3811


... (10 outputs left out)

3400/3811


3600/3811


3800/3811


wrote ../runs/pipeline/submission.csv: 3811 rows, 3811 with detections, max 500 boxes
